In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from dotenv import load_dotenv
import os 

load_dotenv()

True

In [3]:
spark = (
    SparkSession.builder
    .appName("aula_09_jdbc")
    .config("spark.jars", "/opt/jdbc/postgresql.jar")
    .getOrCreate()
)

26/10/08 00:37:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 00:37:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
df_pedidos = spark.read.csv("/workspace/datasets/09_jdbc/pedidos.csv", header= True, inferSchema= True)

In [5]:
df_pedidos.write.mode("overwrite").parquet("/workspace/datasets/09_jdbc/df_pedidos_parquet")

In [6]:
df_pedidos_parquet = spark.read.parquet("/workspace/datasets/09_jdbc/df_pedidos_parquet")

In [7]:
jdbc_url = "jdbc:postgresql://postgres:5432/spark_db"

credencias = {
    "user": "postgres",
    "password": os.getenv("POSTGRES_PASSWORD"),
    "driver": "org.postgresql.Driver"
}

In [ ]:
df_pedidos_parquet.write.jdbc(
    url=jdbc_url,
    table= "public.staging_pedidos",
    mode= "overwrite",
    properties= {**credencias, "" : ""}
)

In [9]:
df_pedidos_postgres = spark.read.jdbc(
    url= jdbc_url,
    table= "public.staging_pedidos",
    properties= credencias
)

In [10]:
df_pedidos_postgres.show(5)

+---------+----------+-----------+---------+-----------+-----------+
|pedido_id|cliente_id|data_pedido|   status|      canal|valor_total|
+---------+----------+-----------+---------+-----------+-----------+
|    10001|       452| 2025-10-09|concluido|       site|     473.56|
|    10002|        71| 2025-08-03|concluido|        app|     303.13|
|    10003|       411| 2025-10-06|  enviado|       loja|     281.64|
|    10004|       184| 2025-09-03|concluido|       loja|     281.65|
|    10005|       290| 2025-09-15|concluido|marketplace|     893.22|
+---------+----------+-----------+---------+-----------+-----------+
only showing top 5 rows



In [11]:
df_pedidos_postgres.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- valor_total: double (nullable = true)



In [12]:
df_pedidos_postgres.count()

5000

In [13]:
df_fato_large = spark.read.jdbc(
    url= jdbc_url,
    table= "public.fato_pedidos_large",
    properties= credencias
)

In [14]:
df_fato_large.count()

2000000

In [15]:
df_fato_large.rdd.getNumPartitions()

1

In [16]:
df_fato_large2 = spark.read.jdbc(
    url= jdbc_url,
    table = "public.fato_pedidos_large",
    column= "pedido_id",
    lowerBound= 1,
    upperBound= 2000000,
    numPartitions= 4,
    properties= credencias
    
)

In [17]:
df_fato_large2.show(1)

+---------+----------+-----------+-------+-----+--------------------+
|pedido_id|cliente_id|data_pedido| status|canal|         valor_total|
+---------+----------+-----------+-------+-----+--------------------+
|        1|         1| 2024-01-01|enviado|  app|51.00000000000000...|
+---------+----------+-----------+-------+-----+--------------------+
only showing top 1 row



In [18]:
df_fato_large2.count()

2000000

In [19]:
df_fato_large2.rdd.getNumPartitions()

4

In [20]:
(
    df_fato_large2.withColumn("spark_partition", F.spark_partition_id())
    .groupby("spark_partition")
    .count()
    .orderBy("spark_partition")
    
).show()

+---------------+------+
|spark_partition| count|
+---------------+------+
|              0|500001|
|              1|499999|
|              2|499999|
|              3|500001|
+---------------+------+



In [21]:
df_filtrado_jdbc = spark.read.jdbc(
    url= jdbc_url,
    table= """( 
        
        SELECT
            pedido_id,
            cliente_id,
            valor_total
        FROM public.fato_pedidos_large
        WHERE status = 'concluido'    
            
    ) AS df_filtrado""",
    column= "pedido_id",
    lowerBound= 1,
    upperBound= 2000000,
    numPartitions= 4,
    properties= credencias
    
)

In [22]:
# lowerBound e upperBound representam a faixa de VALORES da partitionColumn,
# não a quantidade de linhas retornadas pela consulta.

COUNT(*) responde:
"quantas linhas tenho?"

MIN/MAX da partitionColumn respondem:
"qual faixa estou dividindo?"

In [23]:
(
    df_filtrado_jdbc.withColumn("spark_partition", F.spark_partition_id())
    .groupby("spark_partition")
    .count()
    .orderBy("spark_partition")
    
).show()

+---------------+------+
|spark_partition| count|
+---------------+------+
|              0|125000|
|              1|125000|
|              2|124999|
|              3|125001|
+---------------+------+



In [24]:
df_filtrado_jdbc.count()

500000

In [25]:
df_filtrado_jdbc.printSchema()

root
 |-- pedido_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- valor_total: decimal(38,18) (nullable = true)



In [26]:
df_fetch = spark.read.jdbc(
    url= jdbc_url,
    properties= {**credencias, "fetchsize" : "100000"},
    table= """(
        
    SELECT
        pedido_id,
        cliente_id,
        valor_total
    FROM public.fato_pedidos_large
    WHERE status = 'concluido' 
    
    )AS df_fetch_filtrado""",
    column= "pedido_id",
    lowerBound= 1,
    upperBound= 2000000,
    numPartitions= 4
)

In [27]:
df_fetch.count()

500000

In [28]:
df_fetch.rdd.getNumPartitions()

4

In [29]:
df_filtrado_jdbc.rdd.getNumPartitions()

4

In [31]:
df_filtrado_jdbc.write.jdbc(
    url= jdbc_url,
    properties = {**credencias, "batchsize" : "5000"},
    mode= "append",
    table = "public.pedidos_concluidos_batch"
)

In [19]:
spark.stop()